# AIMO3 TIR Baseline

Tool-Integrated Reasoning with vLLM + majority voting.

**Setup:**
- Model: QwQ-32B AWQ (attached as model source → `/kaggle/input/qwq-32b/transformers/qwq-32b-awq/1/`)
- Packages: installed offline from `ermecan/aimo3-dependency-dataset`
- Inference: vLLM with tensor_parallel_size=2 (2× H100)
- Strategy: N=16 samples with TIR, majority vote

In [ ]:
import os

IS_COMPETITION_RERUN = os.getenv("KAGGLE_IS_COMPETITION_RERUN") is not None

MODEL_PATH = "/kaggle/input/models/qwen-lm/qwq-32b/transformers/qwq-32b-awq/1"
N_SAMPLES = 16 if IS_COMPETITION_RERUN else 2
TENSOR_PARALLEL = 2 if IS_COMPETITION_RERUN else 1
LOAD_FORMAT = "auto" if IS_COMPETITION_RERUN else "dummy"

MAX_TIR_STEPS = 4
VLLM_PORT = 8080
VLLM_HOST = f"http://localhost:{VLLM_PORT}/v1"
GPU_MEMORY_UTIL = 0.92
MAX_MODEL_LEN = 8192 if IS_COMPETITION_RERUN else 4096

print(f"Mode: {'COMPETITION' if IS_COMPETITION_RERUN else 'DEV (dummy weights)'}")
print(f"Model: {MODEL_PATH}")
print(
    f"N_SAMPLES={N_SAMPLES}, tensor_parallel={TENSOR_PARALLEL}, load_format={LOAD_FORMAT}"
)

In [ ]:
# ── Install packages from offline dataset (no internet in competition run) ───
# animsamuelk/utils-aimo-3: vLLM 0.16.0 + outlines_core 0.2.11 + all deps
# Built for cp312 (Kaggle's current Python), March 2026
import subprocess
import sys

DEPS_DIR = "/kaggle/input/utils-aimo-3"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--no-index",
        "--find-links",
        DEPS_DIR,
        "vllm",
        "openai",
        "polars",
    ],
    check=True,
)
print("Packages installed")

In [ ]:
# ── Start inference server ───────────────────────────────────────────────────
import json
import os
import subprocess
import sys
import threading
import time
import urllib.request
from http.server import BaseHTTPRequestHandler, HTTPServer

if IS_COMPETITION_RERUN:
    # Real vLLM with QwQ-32B AWQ on 2× H100
    VLLM_LOG = "/kaggle/working/vllm.log"
    vllm_proc = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "vllm.entrypoints.openai.api_server",
            "--model",
            MODEL_PATH,
            "--port",
            str(VLLM_PORT),
            "--tensor-parallel-size",
            str(TENSOR_PARALLEL),
            "--gpu-memory-utilization",
            str(GPU_MEMORY_UTIL),
            "--max-model-len",
            str(MAX_MODEL_LEN),
            "--trust-remote-code",
            "--disable-log-requests",
        ],
        stdout=open(VLLM_LOG, "w"),
        stderr=subprocess.STDOUT,
    )
    print("Waiting for vLLM server...", flush=True)
    for i in range(120):
        try:
            urllib.request.urlopen(f"http://localhost:{VLLM_PORT}/health", timeout=2)
            print(f"vLLM ready after {i * 5}s", flush=True)
            break
        except Exception:
            time.sleep(5)
            if i % 6 == 5:
                try:
                    lines = open(VLLM_LOG).readlines()
                    print(f"[{i * 5}s] {lines[-1].strip()}", flush=True)
                except Exception:
                    pass
    else:
        print("".join(open(VLLM_LOG).readlines()[-30:]))
        raise RuntimeError("vLLM server failed to start")

else:
    # Dev run: mock OpenAI server — no GPU, tests pipeline end-to-end
    MOCK_RESPONSE = json.dumps(
        {
            "id": "mock",
            "object": "chat.completion",
            "choices": [
                {
                    "index": 0,
                    "finish_reason": "stop",
                    "message": {
                        "role": "assistant",
                        "content": "The answer is \\\\boxed{42}.",
                    },
                }
            ],
            "model": MODEL_PATH,
            "usage": {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0},
        }
    ).encode()

    class _MockHandler(BaseHTTPRequestHandler):
        def do_GET(self):
            self.send_response(200)
            self.end_headers()
            self.wfile.write(b"ok")

        def do_POST(self):
            self.rfile.read(int(self.headers.get("Content-Length", 0)))
            self.send_response(200)
            self.send_header("Content-Type", "application/json")
            self.end_headers()
            self.wfile.write(MOCK_RESPONSE)

        def log_message(self, *args):
            pass

    mock_server = HTTPServer(("localhost", VLLM_PORT), _MockHandler)
    threading.Thread(target=mock_server.serve_forever, daemon=True).start()
    print(
        "Mock server started (dev mode — returns \\boxed{42} for all problems)",
        flush=True,
    )

In [ ]:
# ── TIR Solver ──────────────────────────────────────────────────────────────
import os
import re
import subprocess
import sys
import tempfile
from collections import Counter
from typing import Optional

from openai import OpenAI

SYSTEM_PROMPT = """\
You are an expert math competition solver. Solve the problem step by step.

When you need to compute something, write Python code in a ```python block.
You will see the output and can continue reasoning.

End your response with \\boxed{N} where N is your final integer answer (0-99999).
"""

_llm = OpenAI(base_url=VLLM_HOST, api_key="local")


def execute_python(code: str, timeout: int = 15) -> str:
    """Run code in subprocess, block unsafe patterns."""
    for pattern in [
        r"\bos\b",
        r"\bsubprocess\b",
        r"\bopen\b",
        r"__import__",
        r"socket",
        r"urllib",
        r"requests",
    ]:
        if re.search(pattern, code):
            return "[blocked: unsafe code]"
    with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
        f.write("from math import *\nimport sympy\n" + code)
        tmp = f.name
    try:
        r = subprocess.run(
            [sys.executable, tmp], capture_output=True, text=True, timeout=timeout
        )
        out = r.stdout
        if r.returncode != 0 and r.stderr:
            out += f"\n[err]: {r.stderr[:300]}"
        return out.strip() or "(no output)"
    except subprocess.TimeoutExpired:
        return f"[timeout after {timeout}s]"
    finally:
        os.unlink(tmp)


def extract_code_blocks(text: str) -> list:
    return re.findall(r"```python\n(.*?)```", text, re.DOTALL)


def extract_answer(text: str) -> int | None:
    for m in re.finditer(r"\\boxed\{(\d+)\}", text):
        v = int(m.group(1))
        if 0 <= v <= 99999:
            return v
    for m in re.finditer(
        r"(?:answer\s+is|answer:|final\s+answer[:\s]+|=\s*)(\d+)", text, re.IGNORECASE
    ):
        v = int(m.group(1))
        if 0 <= v <= 99999:
            return v
    return None


def _chat(messages: list, temperature: float) -> str:
    r = _llm.chat.completions.create(
        model=MODEL_PATH,
        messages=messages,
        temperature=temperature,
        max_tokens=4096,
    )
    return r.choices[0].message.content or ""


def solve_with_voting(problem: str, n: int = N_SAMPLES) -> int:
    answers = []
    for i in range(n):
        temp = 0.0 if i == 0 else 0.7
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": problem},
        ]
        for _ in range(MAX_TIR_STEPS):
            reply = _chat(messages, temperature=temp)
            messages.append({"role": "assistant", "content": reply})
            code_blocks = extract_code_blocks(reply)
            if code_blocks:
                out = execute_python(code_blocks[-1])
                messages.append({"role": "user", "content": f"Code output:\n{out}"})
            if not code_blocks:
                if extract_answer(reply):
                    break
                messages.append(
                    {"role": "user", "content": "Give your final answer as \\boxed{N}."}
                )
            elif extract_answer(reply):
                break
        all_text = "\n".join(m["content"] for m in messages if m["role"] == "assistant")
        ans = extract_answer(all_text)
        if ans is not None:
            answers.append(ans)
    return Counter(answers).most_common(1)[0][0] if answers else 0


print("Solver ready", flush=True)

In [ ]:
# ── Kaggle gateway integration ───────────────────────────────────────────────
sys.path.insert(
    0, "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3"
)
import kaggle_evaluation.aimo_3_inference_server as aimo3
import polars as pl

TEST_CSV = (
    "/kaggle/input/competitions/ai-mathematical-olympiad-progress-prize-3/test.csv"
)


def predict(df: pl.DataFrame) -> pl.DataFrame:
    """Called once per problem by the competition gateway."""
    row = df.row(0, named=True)
    problem_id = row["id"]
    problem = row["problem"]
    print(f"[{problem_id}] solving...", flush=True)
    answer = solve_with_voting(problem, n=N_SAMPLES)
    print(f"[{problem_id}] answer={answer}", flush=True)
    return pl.DataFrame({"id": [problem_id], "answer": [answer]})


inference_server = aimo3.AIMO3InferenceServer(predict)

if IS_COMPETITION_RERUN:
    # Competition: real gateway connects externally, serve() blocks until done
    inference_server.serve()
else:
    # Dev: bypass run_local_gateway — it unpacks polars DataFrame via *df which
    # iterates column names, causing predict() to receive 2 string args instead
    # of 1 DataFrame. Call predict() directly instead.
    test_df = pl.read_csv(TEST_CSV)
    all_predictions = []
    for row in test_df.iter_slices(n_rows=1):
        result = predict(row)
        all_predictions.append(result)
    submission = pl.concat(all_predictions)
    submission.write_parquet("/kaggle/working/submission.parquet")
    print(f"submission.parquet written ({len(test_df)} rows)", flush=True)
    print(submission)